# GitHub Machine-Learning Repository Analysis

## GitHub Repository

[Open the GitHub repository](https://github.com/omarelsayed0101/level3-project2-machine-learning-repositories)

## Ethics Reflection

### Question 1: Why is it important to verify data collected from public APIs?

Verification is important because a successful API response does not guarantee that every field is present, current, correctly typed, or suitable for analysis. API schemas can change, values can be missing, and records can be duplicated or returned in an unexpected order. Checking the response structure, DataFrame shape, data types, missing values, duplicate records, date formats, and saved-file reload helps reduce the risk of making conclusions from malformed or incomplete data.

### Question 2: Why should data analysts document the source of their data?

Analysts should document the source so that their work is transparent, traceable, and reproducible. A reader should be able to identify the API endpoint, query parameters, collection date, transformation rules, and limitations of the dataset. In this project, the source URL, retrieval timestamp, raw API response, cleaning decisions, and snapshot limitation are recorded so that the analysis can be reviewed and repeated responsibly.

### Question 3: How can missing or inaccurate data affect data analysis and decision-making?

Missing or inaccurate values can distort counts, averages, rankings, language comparisons, and trend charts. For example, a missing programming language can understate the size of another group, an incorrect date can place a repository in the wrong time period, and duplicate records can inflate totals. Inaccurate license information could also create legal or operational risk. Analysts should identify, document, and appropriately handle these issues, then communicate the remaining limitations before using the results to support decisions.

## 1. Install or import dependencies

Run this notebook in Google Colab or Jupyter with `pandas`, `requests`, and `matplotlib` available. The complete workflow code is included below; no separate Python script is required to understand or execute the project.

In [ ]:
from pathlib import Path
import json
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
CSV_PATH = PROJECT_DIR / 'github_projects.csv'
DB_PATH = PROJECT_DIR / 'github_projects.db'
print('Project directory:', PROJECT_DIR)

## 2. Task 1 — Collect, clean, and verify the dataset

The next code cell contains the complete GitHub API collection pipeline: API retrieval, DataFrame creation and exploration, required-column selection, nested owner/license extraction, missing-value handling, duplicate removal, date conversion, column renaming, CSV saving, and reload verification.

In [ ]:
"""Collect and prepare GitHub machine-learning repository data.

This script implements Task 1 of the Level 3 Project 2 rubric.
"""
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
import os

import pandas as pd
import requests


PROJECT_DIR = Path.cwd()  # Notebook working directory
URL = "https://api.github.com/search/repositories?q=machine+learning&sort=stars&order=desc&per_page=100"
OUTPUT_CSV = PROJECT_DIR / "github_projects.csv"
SUMMARY_JSON = PROJECT_DIR / "collection_summary.json"
RAW_JSON = PROJECT_DIR / "github_api_response.json"

REQUIRED_COLUMNS = [
    "name",
    "owner",
    "language",
    "stargazers_count",
    "forks_count",
    "watchers_count",
    "open_issues_count",
    "created_at",
    "updated_at",
    "license",
]


def collect_api_data() -> list[dict]:
    """Retrieve repository records from the GitHub Search API."""
    headers = {
        "Accept": "application/vnd.github+json",
        "User-Agent": "level3-machine-learning-repository-analysis",
    }
    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    if token:
        headers["Authorization"] = f"Bearer {token}"
    response = requests.get(URL, headers=headers, timeout=30)
    response.raise_for_status()
    payload = response.json()
    if "items" not in payload or not isinstance(payload["items"], list):
        raise ValueError("The GitHub response did not contain an items list.")
    RAW_JSON.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return payload["items"]


def prepare_dataset(items: list[dict]) -> tuple[pd.DataFrame, dict]:
    """Convert raw API records into the required clean analysis dataset."""
    raw_df = pd.DataFrame(items)
    print("Initial dataset shape:", raw_df.shape)
    print("Initial columns:", list(raw_df.columns))
    print("Initial data types:\n", raw_df.dtypes)

    missing_required = [column for column in REQUIRED_COLUMNS if column not in raw_df.columns]
    if missing_required:
        raise ValueError(f"Required API columns are missing: {missing_required}")

    df = raw_df[REQUIRED_COLUMNS].copy()
    missing_before = df.isna().sum().to_dict()

    # Extract values from nested API objects before handling missing values or duplicates.
    df["owner"] = df["owner"].apply(
        lambda value: value.get("login") if isinstance(value, dict) else value
    )
    df["license"] = df["license"].apply(
        lambda value: (
            value.get("spdx_id") or value.get("name")
            if isinstance(value, dict)
            else value
        )
    )

    # Explicit, documented missing-value policy for analysis-ready data.
    df["owner"] = df["owner"].fillna("Unknown")
    df["language"] = df["language"].fillna("Unknown")
    df["license"] = df["license"].fillna("No license")
    numeric_columns = ["stargazers_count", "forks_count", "watchers_count", "open_issues_count"]
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce").fillna(0).astype(int)

    # A repository is uniquely identified here by owner and name.
    duplicate_rows_before = int(df.duplicated(subset=["owner", "name"]).sum())
    df = df.drop_duplicates(subset=["owner", "name"], keep="first").copy()
    duplicates_removed = duplicate_rows_before

    # Parse dates and store ISO date strings for portable SQLite/date analysis.
    for column in ["created_at", "updated_at"]:
        df[column] = pd.to_datetime(df[column], errors="coerce", utc=True)
    date_rows_before_drop = len(df)
    df = df.dropna(subset=["created_at", "updated_at"]).copy()
    dropped_invalid_dates = date_rows_before_drop - len(df)
    df["created_at"] = df["created_at"].dt.strftime("%Y-%m-%d")
    df["updated_at"] = df["updated_at"].dt.strftime("%Y-%m-%d")

    df = df.rename(
        columns={
            "stargazers_count": "stars",
            "forks_count": "forks",
            "watchers_count": "watchers",
            "open_issues_count": "open_issues",
            "created_at": "created_date",
            "updated_at": "updated_date",
        }
    )
    final_columns = [
        "name",
        "owner",
        "language",
        "stars",
        "forks",
        "watchers",
        "open_issues",
        "created_date",
        "updated_date",
        "license",
    ]
    df = df[final_columns].reset_index(drop=True)
    missing_after = df.isna().sum().to_dict()
    duplicate_rows_after = int(df.duplicated().sum())

    summary = {
        "source_url": URL,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "raw_records": len(items),
        "cleaned_records": len(df),
        "initial_shape": list(raw_df.shape),
        "missing_values_before": missing_before,
        "missing_values_after": missing_after,
        "duplicate_rows_before": duplicate_rows_before,
        "duplicate_rows_after": duplicate_rows_after,
        "duplicates_removed": duplicates_removed,
        "invalid_date_rows_dropped": dropped_invalid_dates,
        "final_columns": final_columns,
        "missing_value_policy": {
            "owner": "Unknown",
            "language": "Unknown",
            "license": "No license",
            "numeric_metrics": 0,
            "dates": "Rows with invalid or missing dates removed",
        },
    }
    return df, summary


def main() -> None:
    items = collect_api_data()
    df, summary = prepare_dataset(items)
    df.to_csv(OUTPUT_CSV, index=False)

    # Reload verification is intentionally part of the pipeline acceptance check.
    verified_df = pd.read_csv(OUTPUT_CSV)
    if list(verified_df.columns) != summary["final_columns"]:
        raise AssertionError("Reloaded CSV columns do not match the expected schema.")
    if verified_df.empty:
        raise AssertionError("The verified CSV is empty.")
    if int(verified_df.isna().sum().sum()) != 0:
        raise AssertionError("The verified CSV still contains missing values.")

    summary["verification"] = {
        "reloaded_successfully": True,
        "reloaded_shape": list(verified_df.shape),
        "reloaded_missing_values": int(verified_df.isna().sum().sum()),
        "file": OUTPUT_CSV.name,
    }
    SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print("\nCleaned dataset preview:\n", verified_df.head().to_string(index=False))
    print("\nSaved and verified:", OUTPUT_CSV)
    print("Collection summary:", json.dumps(summary, indent=2))


main()


# References:
# GitHub REST API documentation: https://docs.github.com/en/rest/search/search
# Pandas documentation: https://pandas.pydata.org/docs/


In [ ]:
df = pd.read_csv('github_projects.csv')
display(df.head())
print('Shape:', df.shape)
print('Missing values:', int(df.isna().sum().sum()))
print('Duplicate rows:', int(df.duplicated().sum()))

## 3. Task 2 — Store and analyse the data

The next code cell contains the complete SQLite and SQL workflow: database creation, `Repositories` table import, filtering, searching, AND/OR/NOT logic, sorting, limiting, COUNT, AVG, GROUP BY, HAVING, Matplotlib charts, and written interpretation.

In [ ]:
"""Load the cleaned repository data into SQLite, run rubric queries, and create charts."""
from __future__ import annotations

import json
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_DIR = Path.cwd()  # Notebook working directory
CSV_PATH = PROJECT_DIR / "github_projects.csv"
DB_PATH = PROJECT_DIR / "github_projects.db"
RESULTS_PATH = PROJECT_DIR / "sql_results.json"
CHARTS_DIR = PROJECT_DIR / "charts"
CHARTS_DIR.mkdir(exist_ok=True)


def load_database() -> None:
    """Create Repositories and import the prepared CSV."""
    df = pd.read_csv(CSV_PATH)
    with sqlite3.connect(DB_PATH) as connection:
        df.to_sql("Repositories", connection, if_exists="replace", index=False)
        row_count = connection.execute("SELECT COUNT(*) FROM Repositories").fetchone()[0]
    if row_count != len(df):
        raise AssertionError("SQLite row count does not match the cleaned CSV.")


def run_queries() -> dict[str, object]:
    """Execute all required SQL tasks and return JSON-serializable results."""
    with sqlite3.connect(DB_PATH) as connection:
        queries = {
            "more_than_10000_stars": """
                SELECT name, owner, stars, language
                FROM Repositories
                WHERE stars > 10000
                ORDER BY stars DESC
            """,
            "names_containing_machine": """
                SELECT name, owner, stars, language
                FROM Repositories
                WHERE name LIKE '%Machine%'
                   OR name LIKE '%machine%'
                ORDER BY stars DESC
            """,
            # Logical-operator query 1: AND.
            "logical_and": """
                SELECT name, owner, stars, language
                FROM Repositories
                WHERE stars > 10000 AND forks > 1000
                ORDER BY stars DESC
            """,
            # Logical-operator query 2: OR and NOT.
            "logical_or_not": """
                SELECT name, owner, stars, language
                FROM Repositories
                WHERE (language = 'Python' OR language = 'C++')
                  AND NOT license = 'No license'
                ORDER BY stars DESC
            """,
            # A second mixed logical query makes the AND/OR/NOT requirement explicit.
            "logical_mixed": """
                SELECT name, owner, stars, language
                FROM Repositories
                WHERE (stars > 10000 OR forks > 5000)
                  AND NOT name LIKE 'test%'
                ORDER BY stars DESC
            """,
            "top_10_by_stars": """
                SELECT name, owner, stars, forks, language
                FROM Repositories
                ORDER BY stars DESC
                LIMIT 10
            """,
            "total_repository_count": """
                SELECT COUNT(*) AS total_repositories
                FROM Repositories
            """,
            "average_stars": """
                SELECT ROUND(AVG(stars), 2) AS average_stars
                FROM Repositories
            """,
            "language_groups_more_than_5": """
                SELECT language, COUNT(*) AS repository_count,
                       ROUND(AVG(stars), 2) AS average_stars
                FROM Repositories
                GROUP BY language
                HAVING COUNT(*) > 5
                ORDER BY repository_count DESC, language
            """,
            "monthly_creation_trends": """
                SELECT substr(created_date, 1, 7) AS creation_month,
                       COUNT(*) AS repository_count
                FROM Repositories
                GROUP BY creation_month
                ORDER BY creation_month
            """,
        }
        results: dict[str, object] = {}
        for name, query in queries.items():
            frame = pd.read_sql_query(query, connection)
            results[name] = frame.to_dict(orient="records")

        table_info = connection.execute("PRAGMA table_info(Repositories)").fetchall()
        results["database_verification"] = {
            "database": DB_PATH.name,
            "table": "Repositories",
            "row_count": int(connection.execute("SELECT COUNT(*) FROM Repositories").fetchone()[0]),
            "columns": [row[1] for row in table_info],
        }
    RESULTS_PATH.write_text(json.dumps(results, indent=2), encoding="utf-8")
    return results


def create_visualizations(results: dict[str, object]) -> None:
    """Create the two required Matplotlib charts from SQL outputs."""
    plt.style.use("seaborn-v0_8-whitegrid")

    top10 = pd.DataFrame(results["top_10_by_stars"])
    top10 = top10.sort_values("stars", ascending=True)
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(top10["name"], top10["stars"], color="#2563eb")
    ax.set_title("Top 10 Machine-Learning Repositories by GitHub Stars", fontsize=15, weight="bold")
    ax.set_xlabel("Stars")
    ax.set_ylabel("Repository")
    ax.ticklabel_format(axis="x", style="plain")
    for index, value in enumerate(top10["stars"]):
        ax.text(value, index, f" {value:,.0f}", va="center", fontsize=9)
    fig.tight_layout()
    fig.savefig(CHARTS_DIR / "top_10_repositories_by_stars.png", dpi=180)
    plt.close(fig)

    trends = pd.DataFrame(results["monthly_creation_trends"])
    trends["creation_month"] = pd.to_datetime(trends["creation_month"], format="%Y-%m")
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(trends["creation_month"], trends["repository_count"], marker="o", color="#059669", linewidth=2)
    ax.set_title("Machine-Learning Repository Creation Trends", fontsize=15, weight="bold")
    ax.set_xlabel("Creation month")
    ax.set_ylabel("Number of repositories")
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(CHARTS_DIR / "repository_creation_trends.png", dpi=180)
    plt.close(fig)


def write_interpretation(results: dict[str, object]) -> None:
    """Write concise, data-supported findings for the project report."""
    top10 = pd.DataFrame(results["top_10_by_stars"])
    languages = pd.DataFrame(results["language_groups_more_than_5"])
    trends = pd.DataFrame(results["monthly_creation_trends"])
    total = results["total_repository_count"][0]["total_repositories"]
    average = results["average_stars"][0]["average_stars"]
    top_repo = top10.iloc[0]
    peak_month = trends.loc[trends["repository_count"].idxmax()]
    language_sentence = ", ".join(
        f"{row.language} ({int(row.repository_count)})" for row in languages.itertuples()
    )

    text = f"""# Analysis Findings

The cleaned dataset contains **{total} repositories**, and the mean repository popularity is **{average:,.2f} stars**. The most-starred repository in the top-ten result is **{top_repo['name']}**, with **{int(top_repo['stars']):,} stars**. The top-ten chart shows that popularity is concentrated among a small set of highly visible projects rather than distributed evenly across all 100 records.

The language grouping query retained languages with more than five repositories. The qualifying groups are **{language_sentence}**. This indicates that the sample is not language-neutral: a small number of languages account for most of the repositories, which is useful when considering skills, tooling, or documentation priorities.

The creation-trend query identifies **{peak_month['creation_month']}** as the month with the largest number of repositories in the sample, at **{int(peak_month['repository_count'])} repositories**. The line chart should be read as the distribution of creation dates within this API snapshot, not as a complete history of all machine-learning repositories on GitHub.

These findings support a practical business interpretation: star counts can help identify high-visibility projects for further review, while language and creation-date patterns can inform technical ecosystem monitoring. They should not be treated as measures of software quality, security, maintenance quality, or adoption without additional validation.
"""
    (PROJECT_DIR / "analysis_findings.md").write_text(text, encoding="utf-8")


def main() -> None:
    if not CSV_PATH.exists():
        raise FileNotFoundError(f"Run collect_data.py first: {CSV_PATH}")
    load_database()
    results = run_queries()
    create_visualizations(results)
    write_interpretation(results)
    print("SQLite database created:", DB_PATH)
    print("SQL results written:", RESULTS_PATH)
    print("Charts written:", sorted(path.name for path in CHARTS_DIR.glob("*.png")))
    print("Total repositories:", results["total_repository_count"])
    print("Average stars:", results["average_stars"])
    print("Top 10:\n", pd.DataFrame(results["top_10_by_stars"]).to_string(index=False))
    print("Languages with more than 5 repositories:\n", pd.DataFrame(results["language_groups_more_than_5"]).to_string(index=False))


main()


# References:
# SQLite documentation: https://www.sqlite.org/docs.html
# Matplotlib documentation: https://matplotlib.org/stable/


In [ ]:
with open('sql_results.json', encoding='utf-8') as file:
    results = json.load(file)

print('Database verification:', results['database_verification'])
display(pd.DataFrame(results['top_10_by_stars']))
display(pd.DataFrame(results['language_groups_more_than_5']))

## 4. Visualizations and interpretation

The analysis code above creates the required Matplotlib charts and writes the findings to `analysis_findings.md`. The following cell displays both chart files.

In [ ]:
from IPython.display import Image, display
display(Image(filename='charts/top_10_repositories_by_stars.png'))
display(Image(filename='charts/repository_creation_trends.png'))

## 5. Task 3 — Git and GitHub publication

The project was initialized, staged, committed, and published to the repository linked above. The repository contains the source scripts, notebook, cleaned CSV, SQLite database, raw API response, SQL outputs, charts, analysis findings, ethics reflection, and Git/GitHub evidence.

For a local terminal workflow, the commands are:

```bash
git init
git status
git add .
git commit -m "Complete GitHub machine learning repository analysis project"
git remote add origin https://github.com/omarelsayed0101/level3-project2-machine-learning-repositories.git
git branch -M main
git push -u origin main
```

The required screenshots and publication evidence are stored in the repository evidence set.